# Подготовка данных Olist

Цель ноутбука — подготовить исходные таблицы Olist к дальнейшему бизнес-анализу: привести данные к корректным типам, проверить ключевые связи между таблицами, агрегировать данные на уровне заказов и сформировать аналитические таблицы для следующих этапов проекта.

На этом этапе данные очищаются и преобразуются только в той части, которая необходима для создания аналитической базы: обрабатываются даты, объединяются связанные таблицы, рассчитываются базовые признаки заказов, оплат, доставки, товаров и отзывов.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.2f}".format)


PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
raw_files = {
    "orders": "olist_orders_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

missing_files = [
    file_name for file_name in raw_files.values() if not (RAW_DATA_DIR / file_name).exists()
]

if missing_files:
    raise FileNotFoundError(f"Не найдены файлы: {missing_files}")

data = {
    table_name: pd.read_csv(RAW_DATA_DIR / file_name) for table_name, file_name in raw_files.items()
}

orders = data["orders"]
customers = data["customers"]
order_items = data["order_items"]
order_payments = data["order_payments"]
order_reviews = data["order_reviews"]
products = data["products"]
category_translation = data["category_translation"]

pd.DataFrame(
    {
        "table_name": list(data.keys()),
        "rows": [table.shape[0] for table in data.values()],
        "columns": [table.shape[1] for table in data.values()],
    }
)

,table_name,rows,columns
0,orders,99441,8
1,customers,99441,5
2,order_items,112650,7
3,order_payments,103886,5
4,order_reviews,99224,7
5,products,32951,9
6,category_translation,71,2


In [3]:
date_columns = {
    "orders": [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
    ],
    "order_items": [
        "shipping_limit_date",
    ],
    "order_reviews": [
        "review_creation_date",
        "review_answer_timestamp",
    ],
}

date_conversion_check = []

for table_name, columns in date_columns.items():
    table = data[table_name]

    for column in columns:
        missing_before = table[column].isna().sum()

        table[column] = pd.to_datetime(table[column], errors="coerce")

        missing_after = table[column].isna().sum()

        date_conversion_check.append(
            {
                "table_name": table_name,
                "column": column,
                "dtype": table[column].dtype,
                "missing_before": missing_before,
                "missing_after": missing_after,
                "new_missing_after_conversion": missing_after - missing_before,
            }
        )

pd.DataFrame(date_conversion_check)

,table_name,column,dtype,missing_before,missing_after,new_missing_after_conversion
0,orders,order_purchase_timestamp,datetime64[ns],0,0,0
1,orders,order_approved_at,datetime64[ns],160,160,0
2,orders,order_delivered_carrier_date,datetime64[ns],1783,1783,0
3,orders,order_delivered_customer_date,datetime64[ns],2965,2965,0
4,orders,order_estimated_delivery_date,datetime64[ns],0,0,0
5,order_items,shipping_limit_date,datetime64[ns],0,0,0
6,order_reviews,review_creation_date,datetime64[ns],0,0,0
7,order_reviews,review_answer_timestamp,datetime64[ns],0,0,0


In [4]:
products_prepared = products.merge(
    category_translation,
    on="product_category_name",
    how="left",
)

products_prepared["has_category_translation"] = products_prepared[
    "product_category_name_english"
].notna()

products_prepared["product_category_name"] = products_prepared["product_category_name"].fillna(
    "unknown"
)

products_prepared["product_category_name_english"] = products_prepared[
    "product_category_name_english"
].fillna("unknown")

products_prepared_check = pd.DataFrame(
    {
        "metric": [
            "rows_in_products",
            "rows_in_products_prepared",
            "unique_product_id",
            "missing_original_category_after_fill",
            "missing_translated_category_after_fill",
            "products_without_category_translation",
        ],
        "value": [
            products.shape[0],
            products_prepared.shape[0],
            products_prepared["product_id"].nunique(),
            products_prepared["product_category_name"].isna().sum(),
            products_prepared["product_category_name_english"].isna().sum(),
            (~products_prepared["has_category_translation"]).sum(),
        ],
    }
)

products_prepared_check

,metric,value
0,rows_in_products,32951
1,rows_in_products_prepared,32951
2,unique_product_id,32951
3,missing_original_category_after_fill,0
4,missing_translated_category_after_fill,0
5,products_without_category_translation,623


In [5]:
order_items_enriched = order_items.merge(
    products_prepared[
        [
            "product_id",
            "product_category_name",
            "product_category_name_english",
        ]
    ],
    on="product_id",
    how="left",
)

order_items_enriched["product_category_name"] = order_items_enriched[
    "product_category_name"
].fillna("unknown")

order_items_enriched["product_category_name_english"] = order_items_enriched[
    "product_category_name_english"
].fillna("unknown")

order_items_enriched_check = pd.DataFrame(
    {
        "metric": [
            "rows_in_order_items",
            "rows_in_order_items_enriched",
            "unique_order_id",
            "unique_product_id",
            "missing_product_id",
            "missing_original_category",
            "missing_translated_category",
        ],
        "value": [
            order_items.shape[0],
            order_items_enriched.shape[0],
            order_items_enriched["order_id"].nunique(),
            order_items_enriched["product_id"].nunique(),
            order_items_enriched["product_id"].isna().sum(),
            order_items_enriched["product_category_name"].isna().sum(),
            order_items_enriched["product_category_name_english"].isna().sum(),
        ],
    }
)

order_items_enriched_check

,metric,value
0,rows_in_order_items,112650
1,rows_in_order_items_enriched,112650
2,unique_order_id,98666
3,unique_product_id,32951
4,missing_product_id,0
5,missing_original_category,0
6,missing_translated_category,0


In [6]:
order_items_agg = order_items_enriched.groupby("order_id", as_index=False).agg(
    items_count=("order_item_id", "count"),
    unique_products_count=("product_id", "nunique"),
    unique_categories_count=("product_category_name_english", "nunique"),
    products_price=("price", "sum"),
    freight_value=("freight_value", "sum"),
)

order_items_agg["order_items_total_value"] = (
    order_items_agg["products_price"] + order_items_agg["freight_value"]
)

pd.DataFrame(
    {
        "metric": [
            "rows_in_order_items_agg",
            "unique_order_id",
            "missing_order_id",
            "total_products_price",
            "total_freight_value",
            "total_order_items_value",
        ],
        "value": [
            order_items_agg.shape[0],
            order_items_agg["order_id"].nunique(),
            order_items_agg["order_id"].isna().sum(),
            order_items_agg["products_price"].sum(),
            order_items_agg["freight_value"].sum(),
            order_items_agg["order_items_total_value"].sum(),
        ],
    }
)

,metric,value
0,rows_in_order_items_agg,"98,666.00"
1,unique_order_id,"98,666.00"
2,missing_order_id,0.00
3,total_products_price,"13,591,643.70"
4,total_freight_value,"2,251,909.54"
5,total_order_items_value,"15,843,553.24"


In [7]:
main_payment_type = (
    order_payments.sort_values(
        ["order_id", "payment_value", "payment_sequential"],
        ascending=[True, False, True],
    )
    .drop_duplicates("order_id")[["order_id", "payment_type"]]
    .rename(columns={"payment_type": "main_payment_type"})
)

payments_agg = (
    order_payments.groupby("order_id", as_index=False)
    .agg(
        payment_records_count=("payment_sequential", "count"),
        payment_methods_count=("payment_type", "nunique"),
        max_payment_installments=("payment_installments", "max"),
        payment_value=("payment_value", "sum"),
    )
    .merge(main_payment_type, on="order_id", how="left")
)

pd.DataFrame(
    {
        "metric": [
            "rows_in_order_payments",
            "rows_in_payments_agg",
            "unique_order_id",
            "missing_order_id",
            "total_payment_value",
            "missing_main_payment_type",
        ],
        "value": [
            order_payments.shape[0],
            payments_agg.shape[0],
            payments_agg["order_id"].nunique(),
            payments_agg["order_id"].isna().sum(),
            payments_agg["payment_value"].sum(),
            payments_agg["main_payment_type"].isna().sum(),
        ],
    }
)

,metric,value
0,rows_in_order_payments,"103,886.00"
1,rows_in_payments_agg,"99,440.00"
2,unique_order_id,"99,440.00"
3,missing_order_id,0.00
4,total_payment_value,"16,008,872.12"
5,missing_main_payment_type,0.00


In [8]:
reviews_agg = order_reviews.groupby("order_id", as_index=False).agg(
    review_records_count=("review_id", "count"),
    review_score=("review_score", "mean"),
    first_review_creation_date=("review_creation_date", "min"),
    last_review_answer_timestamp=("review_answer_timestamp", "max"),
)

pd.DataFrame(
    {
        "metric": [
            "rows_in_order_reviews",
            "rows_in_reviews_agg",
            "unique_order_id",
            "missing_order_id",
            "missing_review_score",
            "min_review_score",
            "max_review_score",
        ],
        "value": [
            order_reviews.shape[0],
            reviews_agg.shape[0],
            reviews_agg["order_id"].nunique(),
            reviews_agg["order_id"].isna().sum(),
            reviews_agg["review_score"].isna().sum(),
            reviews_agg["review_score"].min(),
            reviews_agg["review_score"].max(),
        ],
    }
)

,metric,value
0,rows_in_order_reviews,"99,224.00"
1,rows_in_reviews_agg,"98,673.00"
2,unique_order_id,"98,673.00"
3,missing_order_id,0.00
4,missing_review_score,0.00
5,min_review_score,1.00
6,max_review_score,5.00


In [9]:
orders_base = (
    orders.merge(
        customers[
            [
                "customer_id",
                "customer_unique_id",
                "customer_city",
                "customer_state",
            ]
        ],
        on="customer_id",
        how="left",
    )
    .merge(order_items_agg, on="order_id", how="left")
    .merge(payments_agg, on="order_id", how="left")
    .merge(reviews_agg, on="order_id", how="left")
)

orders_base["order_purchase_date"] = orders_base["order_purchase_timestamp"].dt.normalize()

orders_base["order_purchase_month"] = (
    orders_base["order_purchase_timestamp"].dt.to_period("M").dt.to_timestamp()
)

orders_base["delivery_days"] = (
    orders_base["order_delivered_customer_date"] - orders_base["order_purchase_timestamp"]
).dt.total_seconds() / 86_400

orders_base["estimated_delivery_days"] = (
    orders_base["order_estimated_delivery_date"] - orders_base["order_purchase_timestamp"]
).dt.total_seconds() / 86_400

orders_base["delivery_delay_days"] = (
    orders_base["order_delivered_customer_date"] - orders_base["order_estimated_delivery_date"]
).dt.total_seconds() / 86_400

orders_base["is_delivered_late"] = (
    orders_base["delivery_delay_days"]
    .gt(0)
    .where(orders_base["order_delivered_customer_date"].notna())
)

orders_base["payment_items_diff"] = (
    orders_base["payment_value"] - orders_base["order_items_total_value"]
)

pd.DataFrame(
    {
        "metric": [
            "rows_in_orders",
            "rows_in_orders_analytical_base",
            "unique_order_id",
            "missing_customer_unique_id",
            "missing_items_data",
            "missing_payments_data",
            "missing_reviews_data",
            "orders_with_payment_items_diff",
            "max_abs_payment_items_diff",
        ],
        "value": [
            orders.shape[0],
            orders_base.shape[0],
            orders_base["order_id"].nunique(),
            orders_base["customer_unique_id"].isna().sum(),
            orders_base["items_count"].isna().sum(),
            orders_base["payment_value"].isna().sum(),
            orders_base["review_score"].isna().sum(),
            orders_base["payment_items_diff"].abs().gt(0.01).sum(),
            orders_base["payment_items_diff"].abs().max(),
        ],
    }
)

,metric,value
0,rows_in_orders,"99,441.00"
1,rows_in_orders_analytical_base,"99,441.00"
2,unique_order_id,"99,441.00"
3,missing_customer_unique_id,0.00
4,missing_items_data,775.00
5,missing_payments_data,1.00
6,missing_reviews_data,768.00
7,orders_with_payment_items_diff,381.00
8,max_abs_payment_items_diff,182.81


In [10]:
orders_base["is_delivered"] = orders_base["order_status"].eq("delivered")
orders_base["has_items_data"] = orders_base["items_count"].notna()
orders_base["has_payment_data"] = orders_base["payment_value"].notna()

orders_base["is_valid_sales_order"] = (
    orders_base["is_delivered"] & orders_base["has_items_data"] & orders_base["has_payment_data"]
)

orders_analytical_base = orders_base.copy()

valid_sales_orders = orders_analytical_base[orders_analytical_base["is_valid_sales_order"]].copy()

pd.DataFrame(
    {
        "metric": [
            "rows_in_orders_analytical_base",
            "delivered_orders",
            "orders_with_items_data",
            "orders_with_payment_data",
            "valid_sales_orders",
            "share_of_valid_sales_orders",
        ],
        "value": [
            orders_analytical_base.shape[0],
            orders_analytical_base["is_delivered"].sum(),
            orders_analytical_base["has_items_data"].sum(),
            orders_analytical_base["has_payment_data"].sum(),
            valid_sales_orders.shape[0],
            valid_sales_orders.shape[0] / orders_analytical_base.shape[0],
        ],
    }
)

,metric,value
0,rows_in_orders_analytical_base,"99,441.00"
1,delivered_orders,"96,478.00"
2,orders_with_items_data,"98,666.00"
3,orders_with_payment_data,"99,440.00"
4,valid_sales_orders,"96,477.00"
5,share_of_valid_sales_orders,0.97


In [11]:
valid_sales_orders = valid_sales_orders.sort_values(
    ["customer_unique_id", "order_purchase_timestamp", "order_id"]
).copy()

valid_sales_orders["customer_order_number"] = (
    valid_sales_orders.groupby("customer_unique_id").cumcount() + 1
)

valid_sales_orders["customer_type"] = np.where(
    valid_sales_orders["customer_order_number"].eq(1),
    "новый клиент",
    "повторный клиент",
)

pd.DataFrame(
    {
        "metric": [
            "valid_sales_orders",
            "unique_customers",
            "new_customer_orders",
            "repeat_customer_orders",
            "customers_with_repeat_orders",
        ],
        "value": [
            valid_sales_orders.shape[0],
            valid_sales_orders["customer_unique_id"].nunique(),
            valid_sales_orders["customer_type"].eq("новый клиент").sum(),
            valid_sales_orders["customer_type"].eq("повторный клиент").sum(),
            (valid_sales_orders.groupby("customer_unique_id")["order_id"].nunique().gt(1).sum()),
        ],
    }
)

,metric,value
0,valid_sales_orders,96477
1,unique_customers,93357
2,new_customer_orders,93357
3,repeat_customer_orders,3120
4,customers_with_repeat_orders,2801


## Клиентская сегментация

Классический RFM-анализ в проекте не используется напрямую, потому что большинство клиентов Olist сделали только один заказ. В такой ситуации показатель частоты покупок плохо разделяет клиентов, а квантильная сегментация по частоте заказов (frequency) была бы искусственной.

Вместо этого используется адаптированная бизнес-сегментация по трём признакам:

- общей сумме покупок клиента;
- давности последней покупки;
- факту повторных заказов.

Такой подход позволяет отдельно выделить повторных клиентов и ценных разовых покупателей, которые могут быть перспективны для повторных продаж.


In [12]:
analysis_date = valid_sales_orders["order_purchase_timestamp"].max() + pd.Timedelta(days=1)

customers_agg = valid_sales_orders.groupby("customer_unique_id", as_index=False).agg(
    first_order_timestamp=("order_purchase_timestamp", "min"),
    last_order_timestamp=("order_purchase_timestamp", "max"),
    orders_count=("order_id", "nunique"),
    total_payment_value=("payment_value", "sum"),
    avg_order_value=("payment_value", "mean"),
    avg_review_score=("review_score", "mean"),
    avg_delivery_days=("delivery_days", "mean"),
)

customers_agg["recency_days"] = (analysis_date - customers_agg["last_order_timestamp"]).dt.days

customers_agg["customer_lifetime_days"] = (
    customers_agg["last_order_timestamp"] - customers_agg["first_order_timestamp"]
).dt.days

customers_agg["has_repeat_orders"] = customers_agg["orders_count"].gt(1)

pd.DataFrame(
    {
        "metric": [
            "rows_in_customers_agg",
            "unique_customers",
            "total_orders_in_customers_agg",
            "valid_sales_orders",
            "total_payment_value_in_customers_agg",
            "total_payment_value_in_valid_sales_orders",
            "customers_with_repeat_orders",
        ],
        "value": [
            customers_agg.shape[0],
            customers_agg["customer_unique_id"].nunique(),
            customers_agg["orders_count"].sum(),
            valid_sales_orders.shape[0],
            customers_agg["total_payment_value"].sum(),
            valid_sales_orders["payment_value"].sum(),
            customers_agg["has_repeat_orders"].sum(),
        ],
    }
)

,metric,value
0,rows_in_customers_agg,"93,357.00"
1,unique_customers,"93,357.00"
2,total_orders_in_customers_agg,"96,477.00"
3,valid_sales_orders,"96,477.00"
4,total_payment_value_in_customers_agg,"15,422,461.77"
5,total_payment_value_in_valid_sales_orders,"15,422,461.77"
6,customers_with_repeat_orders,"2,801.00"


In [13]:
customers_agg["orders_count"].value_counts(normalize=True).sort_index().head(10)

orders_count
1    0.97
2    0.03
3    0.00
4    0.00
5    0.00
6    0.00
7    0.00
9    0.00
15   0.00
Name: proportion, dtype: float64

In [14]:
customers_segmented = customers_agg.copy()

customers_segmented["monetary_score"] = pd.qcut(
    customers_segmented["total_payment_value"].rank(method="first"),
    q=4,
    labels=[1, 2, 3, 4],
).astype(int)

customers_segmented["recency_score"] = pd.qcut(
    customers_segmented["recency_days"].rank(method="first", ascending=False),
    q=4,
    labels=[1, 2, 3, 4],
).astype(int)

customers_segmented["repeat_order_group"] = np.where(
    customers_segmented["has_repeat_orders"],
    "повторные клиенты",
    "клиенты с одним заказом",
)

segment_conditions = [
    customers_segmented["has_repeat_orders"] & customers_segmented["monetary_score"].ge(3),
    customers_segmented["has_repeat_orders"],
    customers_segmented["monetary_score"].eq(4) & customers_segmented["recency_score"].ge(3),
    customers_segmented["monetary_score"].eq(4),
    customers_segmented["monetary_score"].ge(3),
]

segment_names = [
    "ценные повторные клиенты",
    "остальные повторные клиенты",
    "ценные недавние разовые клиенты",
    "ценные давние разовые клиенты",
    "разовые клиенты средней ценности",
]

customers_segmented["customer_segment"] = np.select(
    segment_conditions,
    segment_names,
    default="разовые клиенты низкой текущей ценности",
)

customer_segments_check = (
    customers_segmented.groupby("customer_segment", as_index=False)
    .agg(
        customers_count=("customer_unique_id", "nunique"),
        orders_count=("orders_count", "sum"),
        total_payment_value=("total_payment_value", "sum"),
        avg_customer_value=("total_payment_value", "mean"),
        avg_orders_count=("orders_count", "mean"),
        avg_recency_days=("recency_days", "mean"),
    )
    .sort_values("total_payment_value", ascending=False)
)

customer_segments_check["revenue_share"] = (
    customer_segments_check["total_payment_value"]
    / customer_segments_check["total_payment_value"].sum()
)

customer_segments_check

,customer_segment,customers_count,orders_count,total_payment_value,avg_customer_value,avg_orders_count,avg_recency_days,revenue_share
4,ценные недавние разовые клиенты,10868,10868,"4,230,400.04",389.25,1.00,111.53,0.27
3,ценные давние разовые клиенты,10700,10700,"4,198,523.20",392.39,1.00,365.59,0.27
2,разовые клиенты средней ценности,22643,22643,"3,189,180.75",140.85,1.00,234.31,0.21
1,разовые клиенты низкой текущей ценности,46345,46345,"2,940,000.57",63.44,1.00,240.94,0.19
5,ценные повторные клиенты,2467,5245,"836,793.97",339.19,2.13,217.60,0.05
0,остальные повторные клиенты,334,676,"27,563.24",82.52,2.02,240.18,0.00


In [15]:
segment_columns = [
    "customer_segment",
    "monetary_score",
    "recency_score",
    "repeat_order_group",
]

valid_sales_orders = valid_sales_orders.drop(
    columns=segment_columns,
    errors="ignore",
)

valid_sales_orders = valid_sales_orders.merge(
    customers_segmented[
        [
            "customer_unique_id",
            "customer_segment",
            "monetary_score",
            "recency_score",
            "repeat_order_group",
        ]
    ],
    on="customer_unique_id",
    how="left",
)

pd.DataFrame(
    {
        "metric": [
            "valid_sales_orders",
            "unique_order_id",
            "missing_customer_segment",
            "missing_monetary_score",
            "missing_recency_score",
            "missing_repeat_order_group",
        ],
        "value": [
            valid_sales_orders.shape[0],
            valid_sales_orders["order_id"].nunique(),
            valid_sales_orders["customer_segment"].isna().sum(),
            valid_sales_orders["monetary_score"].isna().sum(),
            valid_sales_orders["recency_score"].isna().sum(),
            valid_sales_orders["repeat_order_group"].isna().sum(),
        ],
    }
)

,metric,value
0,valid_sales_orders,96477
1,unique_order_id,96477
2,missing_customer_segment,0
3,missing_monetary_score,0
4,missing_recency_score,0
5,missing_repeat_order_group,0


In [16]:
valid_order_columns = [
    "order_id",
    "customer_unique_id",
    "order_purchase_timestamp",
    "order_purchase_month",
    "customer_type",
    "customer_segment",
    "review_score",
    "delivery_days",
    "is_delivered_late",
]

order_categories_base = (
    order_items_enriched.merge(
        valid_sales_orders[valid_order_columns],
        on="order_id",
        how="inner",
    )
    .groupby(
        [
            "order_id",
            "customer_unique_id",
            "order_purchase_timestamp",
            "order_purchase_month",
            "customer_type",
            "customer_segment",
            "product_category_name",
            "product_category_name_english",
        ],
        as_index=False,
    )
    .agg(
        category_items_count=("order_item_id", "count"),
        category_products_price=("price", "sum"),
        category_freight_value=("freight_value", "sum"),
        review_score=("review_score", "mean"),
        delivery_days=("delivery_days", "mean"),
        is_delivered_late=("is_delivered_late", "max"),
    )
)

order_categories_base["category_total_value"] = (
    order_categories_base["category_products_price"]
    + order_categories_base["category_freight_value"]
)

pd.DataFrame(
    {
        "metric": [
            "rows_in_order_categories_base",
            "unique_order_id",
            "valid_sales_orders",
            "unique_categories",
            "total_category_products_price",
            "total_products_price_in_valid_orders",
        ],
        "value": [
            order_categories_base.shape[0],
            order_categories_base["order_id"].nunique(),
            valid_sales_orders["order_id"].nunique(),
            order_categories_base["product_category_name_english"].nunique(),
            order_categories_base["category_products_price"].sum(),
            valid_sales_orders["products_price"].sum(),
        ],
    }
)

,metric,value
0,rows_in_order_categories_base,"97,275.00"
1,unique_order_id,"96,477.00"
2,valid_sales_orders,"96,477.00"
3,unique_categories,72.00
4,total_category_products_price,"13,221,363.14"
5,total_products_price_in_valid_orders,"13,221,363.14"


In [17]:
delivery_reviews_base = valid_sales_orders[
    valid_sales_orders["order_delivered_customer_date"].notna()
    & valid_sales_orders["review_score"].notna()
].copy()

delivery_reviews_base["delivery_status"] = np.where(
    delivery_reviews_base["delivery_delay_days"].gt(0),
    "доставлен с задержкой",
    "доставлен без задержки",
)

delivery_reviews_base["delivery_speed_group"] = pd.cut(
    delivery_reviews_base["delivery_days"],
    bins=[0, 3, 7, 14, 30, np.inf],
    labels=[
        "до 3 дней",
        "4–7 дней",
        "8–14 дней",
        "15–30 дней",
        "более 30 дней",
    ],
    include_lowest=True,
)

pd.DataFrame(
    {
        "metric": [
            "rows_in_delivery_reviews_base",
            "unique_order_id",
            "missing_delivery_days",
            "missing_delivery_delay_days",
            "missing_review_score",
            "missing_delivery_speed_group",
            "late_orders",
            "not_late_orders",
            "avg_delivery_days",
            "avg_review_score",
        ],
        "value": [
            delivery_reviews_base.shape[0],
            delivery_reviews_base["order_id"].nunique(),
            delivery_reviews_base["delivery_days"].isna().sum(),
            delivery_reviews_base["delivery_delay_days"].isna().sum(),
            delivery_reviews_base["review_score"].isna().sum(),
            delivery_reviews_base["delivery_speed_group"].isna().sum(),
            delivery_reviews_base["delivery_status"].eq("доставлен с задержкой").sum(),
            delivery_reviews_base["delivery_status"].eq("доставлен без задержки").sum(),
            delivery_reviews_base["delivery_days"].mean(),
            delivery_reviews_base["review_score"].mean(),
        ],
    }
)

,metric,value
0,rows_in_delivery_reviews_base,"95,823.00"
1,unique_order_id,"95,823.00"
2,missing_delivery_days,0.00
3,missing_delivery_delay_days,0.00
4,missing_review_score,0.00
5,missing_delivery_speed_group,0.00
6,late_orders,"7,660.00"
7,not_late_orders,"88,163.00"
8,avg_delivery_days,12.52
9,avg_review_score,4.16


In [18]:
prepared_tables = {
    "orders_analytical_base": orders_analytical_base,
    "valid_sales_orders": valid_sales_orders,
    "products_prepared": products_prepared,
    "order_items_enriched": order_items_enriched,
    "order_items_agg": order_items_agg,
    "payments_agg": payments_agg,
    "reviews_agg": reviews_agg,
    "customers_segmented": customers_segmented,
    "order_categories_base": order_categories_base,
    "delivery_reviews_base": delivery_reviews_base,
}

table_key_columns = {
    "orders_analytical_base": ["order_id"],
    "valid_sales_orders": ["order_id"],
    "products_prepared": ["product_id"],
    "order_items_enriched": ["order_id", "order_item_id"],
    "order_items_agg": ["order_id"],
    "payments_agg": ["order_id"],
    "reviews_agg": ["order_id"],
    "customers_segmented": ["customer_unique_id"],
    "order_categories_base": [
        "order_id",
        "product_category_name",
        "product_category_name_english",
    ],
    "delivery_reviews_base": ["order_id"],
}

final_checks = []

for table_name, table in prepared_tables.items():
    key_columns = table_key_columns[table_name]

    if len(key_columns) == 1:
        key = key_columns[0]
        unique_key_values = table[key].nunique()
        missing_key_values = table[key].isna().sum()
        duplicated_key_values = table[key].duplicated().sum()
    else:
        unique_key_values = table[key_columns].drop_duplicates().shape[0]
        missing_key_values = table[key_columns].isna().any(axis=1).sum()
        duplicated_key_values = table[key_columns].duplicated().sum()

    final_checks.append(
        {
            "table_name": table_name,
            "rows": table.shape[0],
            "columns": table.shape[1],
            "key": " + ".join(key_columns),
            "unique_key_values": unique_key_values,
            "missing_key_values": missing_key_values,
            "duplicated_key_values": duplicated_key_values,
        }
    )

pd.DataFrame(final_checks)

,table_name,rows,columns,key,unique_key_values,missing_key_values,duplicated_key_values
0,orders_analytical_base,99441,37,order_id,99441,0,0
1,valid_sales_orders,96477,43,order_id,96477,0,0
2,products_prepared,32951,11,product_id,32951,0,0
3,order_items_enriched,112650,9,order_id + order_item_id,112650,0,0
4,order_items_agg,98666,7,order_id,98666,0,0
5,payments_agg,99440,6,order_id,99440,0,0
6,reviews_agg,98673,5,order_id,98673,0,0
7,customers_segmented,93357,15,customer_unique_id,93357,0,0
8,order_categories_base,97275,15,order_id + product_category_name + product_cat...,97275,0,0
9,delivery_reviews_base,95823,45,order_id,95823,0,0


In [19]:
processed_files = {
    "orders_analytical_base": "orders_analytical_base.csv",
    "valid_sales_orders": "valid_sales_orders.csv",
    "products_prepared": "products_prepared.csv",
    "order_items_enriched": "order_items_enriched.csv",
    "order_items_agg": "order_items_agg.csv",
    "payments_agg": "payments_agg.csv",
    "reviews_agg": "reviews_agg.csv",
    "customers_segmented": "customers_segmented.csv",
    "order_categories_base": "order_categories_base.csv",
    "delivery_reviews_base": "delivery_reviews_base.csv",
}

obsolete_files = [
    "orders_base.csv",
]

for file_name in obsolete_files:
    file_path = PROCESSED_DATA_DIR / file_name

    if file_path.exists():
        file_path.unlink()

saved_files = []

for table_name, file_name in processed_files.items():
    file_path = PROCESSED_DATA_DIR / file_name
    prepared_tables[table_name].to_csv(file_path, index=False)

    saved_files.append(
        {
            "table_name": table_name,
            "file_path": file_path.relative_to(PROJECT_ROOT).as_posix(),
            "rows": prepared_tables[table_name].shape[0],
            "columns": prepared_tables[table_name].shape[1],
        }
    )

pd.DataFrame(saved_files)

,table_name,file_path,rows,columns
0,orders_analytical_base,data/processed/orders_analytical_base.csv,99441,37
1,valid_sales_orders,data/processed/valid_sales_orders.csv,96477,43
2,products_prepared,data/processed/products_prepared.csv,32951,11
3,order_items_enriched,data/processed/order_items_enriched.csv,112650,9
4,order_items_agg,data/processed/order_items_agg.csv,98666,7
5,payments_agg,data/processed/payments_agg.csv,99440,6
6,reviews_agg,data/processed/reviews_agg.csv,98673,5
7,customers_segmented,data/processed/customers_segmented.csv,93357,15
8,order_categories_base,data/processed/order_categories_base.csv,97275,15
9,delivery_reviews_base,data/processed/delivery_reviews_base.csv,95823,45


## Результаты подготовки данных

По результатам подготовки данных:

- Исходные таблицы Olist загружены и приведены к формату, удобному для дальнейшего анализа.
- Даты заказов, доставки, товарных позиций и отзывов преобразованы в формат `datetime`.
- Таблица товаров объединена с переводом категорий, отсутствующие названия категорий обработаны как `unknown`.
- Товарные позиции, оплаты и отзывы агрегированы до уровня заказа, чтобы избежать дублирования строк при объединении таблиц.
- Сформирована основная аналитическая база `orders_analytical_base`, где одна строка соответствует одному заказу.
- Для анализа продаж выделена таблица `valid_sales_orders`, включающая доставленные заказы с данными о товарах и оплатах.
- Сформированы промежуточные таблицы `products_prepared`, `order_items_enriched`, `order_items_agg`, `payments_agg` и `reviews_agg`, которые фиксируют логику подготовки данных.
- Для анализа клиентской базы сформирована таблица `customers_segmented`.
- Для анализа товарных категорий создана таблица `order_categories_base`.
- Для анализа связи доставки и оценок создана таблица `delivery_reviews_base`.
- Финальная проверка подтвердила уровень данных и отсутствие дубликатов по ключам подготовленных таблиц.
- Подготовленные таблицы сохранены в папке `data/processed` и будут использоваться в SQL-части и следующих аналитических этапах проекта.
